# Hands-On: Deep Learning Applications for Resource Optimization
### Efficient Training, Compression, and Deployment of Neural Networks

**Faculty Development Programme**
**Date:** 30 July 2026 · **Time:** 11:30 AM – 1:00 PM

**Presented by:** Abhishek Kumar Shukla
**Senior AI Developer, UniConverge Technologies Pvt. Ltd.**

---

### What we will build today

Two models, three optimization techniques, one continuous story:

1. **Model 1 — trained from scratch:** a tiny character-level Transformer, trained live on toy text.
   Purpose: see the full training loop mechanics end to end, on something small enough to actually
   watch converge in a couple of minutes.
2. **Model 2 — GPT-2 (124M), fine-tuned with LoRA:** a real, publicly available pretrained model,
   adapted to a specific text style using parameter-efficient fine-tuning (PEFT).
3. **Then:** we take Model 2 through **quantization** and **ONNX export**, and directly compare model
   size and inference speed before and after each optimization.

### Session flow (~90 minutes, following the lecture half)

| Time | Section |
|---|---|
| 11:30 – 11:38 | 1. Setup & GPU check |
| 11:38 – 11:55 | 2. Model 1 — tiny Transformer trained from scratch |
| 11:55 – 12:05 | 3. Load GPT-2 and inspect its full parameter count |
| 12:05 – 12:25 | 4. Model 2 — fine-tune GPT-2 with LoRA |
| 12:25 – 12:35 | 5. Quantization — compress the fine-tuned model |
| 12:35 – 12:50 | 6. ONNX export — portable, optimized inference |
| 12:50 – 1:00 | 7. Benchmark comparison + wrap-up |

> **Before we start:** Go to `Runtime` → `Change runtime type` → select **T4 GPU** → `Save`.
> Everything in this notebook is designed to run on Colab's free T4 GPU tier.

---

## 1. Setup & GPU Check

Install the libraries we need for today: PyTorch (already in Colab), Hugging Face `transformers`
(pretrained models), `peft` (LoRA), `optimum-onnx` (ONNX export), and `onnxruntime` (running exported
models).

In [ ]:
# Install the libraries this notebook needs beyond Colab's defaults.
# Run this cell FIRST, before any imports below.
!pip install -q -U transformers peft accelerate
!pip install -q -U optimum-onnx onnxruntime onnx

print("Packages installed.")

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = torch.device("cuda")
    print("\n✅ GPU is ready. Let's go.")
else:
    device = torch.device("cpu")
    print("\n⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then Runtime → Restart session.")
    print("This notebook will still run on CPU, just considerably slower for the LoRA fine-tuning section.")

torch.manual_seed(42)
print("\nDevice for this session:", device)

## 2. Model 1 — A Tiny Transformer, Trained From Scratch

Before touching a 124-million-parameter pretrained model, let's build and train the smallest possible
version of the same idea — a character-level Transformer with only a few thousand parameters — so the
full training loop (forward pass → loss → backward pass → optimizer step) is fully visible and fast
enough to watch converge live.

We'll train it to predict the next character in a short piece of text.

In [ ]:
# A small piece of text to train on -- character-level, so the vocabulary is tiny (just the
# unique characters in this string) and training is extremely fast.
text = (
    "deep learning models can be optimized for speed and memory using techniques "
    "such as quantization pruning distillation and low rank adaptation these methods "
    "make large models practical to deploy on real world constrained hardware "
) * 20  # repeat so there's enough data for a few training steps

chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Unique characters (vocabulary):", chars)
print("Vocabulary size:", vocab_size)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

data = torch.tensor([char_to_idx[ch] for ch in text], dtype=torch.long)
print("Encoded data shape:", data.shape)
print("First 20 encoded characters:", data[:20].tolist())
print("Decoded back:", ''.join(idx_to_char[i.item()] for i in data[:20]))

In [ ]:
BLOCK_SIZE = 32   # how many characters of context the model sees at once
BATCH_SIZE = 16

def get_batch(data, block_size, batch_size):
    """Sample random (context, target) pairs from the text for training.
    Target is always the context shifted by one character -- 'predict the next character'."""
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch(data, BLOCK_SIZE, BATCH_SIZE)
print("One batch of inputs, shape:", xb.shape, "(batch_size, block_size)")
print("One batch of targets, shape:", yb.shape)
print("\nExample input sequence: ", ''.join(idx_to_char[i.item()] for i in xb[0]))
print("Corresponding target:    ", ''.join(idx_to_char[i.item()] for i in yb[0]))
print("(Notice the target is the input shifted one character to the left)")

### Building the tiny Transformer

This mirrors the same core Transformer block covered in the lecture — token embedding, position
embedding, self-attention, and a small feed-forward network — just scaled down to a handful of
parameters instead of hundreds of millions.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

EMBED_DIM = 32
NUM_HEADS = 2
NUM_LAYERS = 2

class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, block_size, num_heads, num_layers):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(block_size, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_head = nn.Linear(embed_dim, vocab_size)
        self.block_size = block_size

    def forward(self, idx):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)                                   # (B, T, embed_dim)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device)) # (T, embed_dim)
        x = tok_emb + pos_emb

        # Causal mask: position i can only attend to positions <= i (can't see the future)
        causal_mask = torch.triu(torch.ones(T, T, device=idx.device) * float('-inf'), diagonal=1)
        x = self.transformer(x, mask=causal_mask)

        logits = self.output_head(x)   # (B, T, vocab_size)
        return logits

tiny_model = TinyTransformer(vocab_size, EMBED_DIM, BLOCK_SIZE, NUM_HEADS, NUM_LAYERS).to(device)
tiny_param_count = sum(p.numel() for p in tiny_model.parameters())
print(f"Tiny Transformer parameter count: {tiny_param_count:,}")
print("(Compare this to GPT-2's 124 million parameters, coming up in Section 3)")

### The training loop — visible, end to end

This is the same four-step loop underneath every neural network in this session, including the much
larger GPT-2 model we fine-tune next: **forward pass → compute loss → backward pass → optimizer step.**

In [ ]:
optimizer = torch.optim.AdamW(tiny_model.parameters(), lr=1e-3)

TRAIN_STEPS = 500
tiny_model.train()

losses = []
for step in range(TRAIN_STEPS):
    xb, yb = get_batch(data, BLOCK_SIZE, BATCH_SIZE)
    xb, yb = xb.to(device), yb.to(device)

    # 1. Forward pass
    logits = tiny_model(xb)

    # 2. Compute loss (cross-entropy between predicted and actual next character)
    B, T, V = logits.shape
    loss = F.cross_entropy(logits.view(B*T, V), yb.view(B*T))

    # 3. Backward pass
    optimizer.zero_grad()
    loss.backward()

    # 4. Optimizer step
    optimizer.step()

    losses.append(loss.item())
    if step % 50 == 0:
        print(f"Step {step:4d} | Loss: {loss.item():.4f}")

print(f"\nFinal loss: {losses[-1]:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(losses)
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("Tiny Transformer training loss (should trend downward)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Generate text from the trained tiny model -- start with a seed character and predict forward
@torch.no_grad()
def generate(model, start_text, num_chars, block_size):
    model.eval()
    idx = torch.tensor([[char_to_idx[ch] for ch in start_text]], dtype=torch.long, device=device)
    for _ in range(num_chars):
        idx_cond = idx[:, -block_size:]   # only keep the last block_size characters as context
        logits = model(idx_cond)
        logits = logits[:, -1, :]          # take the prediction for the next character only
        probs = F.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_idx], dim=1)
    return ''.join(idx_to_char[i.item()] for i in idx[0])

generated = generate(tiny_model, start_text="deep ", num_chars=100, block_size=BLOCK_SIZE)
print("Generated text:")
print(generated)
print("\n(With only", f"{tiny_param_count:,}", "parameters and 500 training steps on repeated text,")
print("expect something that captures local patterns but not full coherence -- that's expected.)")

### What this demonstrated

- The **exact same four-step training loop** (forward → loss → backward → optimizer step) that trains
  every neural network in this session, GPT-2 included
- **Causal masking** — the mechanism that prevents a language model from "cheating" by looking at
  future tokens, which is exactly what makes it a *generative* model
- A concrete sense of scale: this model's parameter count versus GPT-2's, coming up next

We now move to a real, pretrained 124-million-parameter model and adapt it efficiently — rather than
training something this size from scratch, which would need far more data, time, and compute than a
90-minute session allows.

## 3. Load GPT-2 and Inspect Its Full Parameter Count

GPT-2 (small, 124M parameters) is small enough to fine-tune quickly, well documented, and — critically
for today — has clean, well-tested LoRA support in the `peft` library.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token   # GPT-2 has no pad token by default; reuse end-of-text

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

total_params = sum(p.numel() for p in base_model.parameters())
print(f"GPT-2 total parameters: {total_params:,}")
print(f"That is {total_params / tiny_param_count:,.0f}x more parameters than our tiny Transformer.")

In [ ]:
# Quick sanity check: does the base model generate coherent text at all?
# (It should -- it's already pretrained on a huge general text corpus.)
base_model.eval()
prompt = "The future of artificial intelligence is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output = base_model.generate(
        **inputs, max_new_tokens=40, do_sample=True, temperature=0.8,
        pad_token_id=tokenizer.eos_token_id,
    )

print("Base GPT-2 output:")
print(tokenizer.decode(output[0], skip_special_tokens=True))

### Why we don't fully fine-tune this model

Recall from the lecture: full fine-tuning of a 124M-parameter model with the Adam optimizer requires
storing roughly 16 bytes per parameter (weights + gradients + optimizer state) — around **2 GB** just
for this relatively small model, before counting activations. LoRA will let us adapt this model while
touching only a small fraction of that.

## 4. Model 2 — Fine-Tune GPT-2 With LoRA

We'll freeze the entire base GPT-2 model and attach small, trainable LoRA layers to its attention
blocks. GPT-2's attention module is named `c_attn` internally (a combined query/key/value projection),
which is what we target.

In [ ]:
# Install the libraries this notebook needs beyond Colab's defaults.
# Run this cell FIRST, before any imports below.
#
# IMPORTANT: Colab ships an older torchao by default (0.10.0). peft checks for
# torchao>=0.16.0 the moment it's imported -- even in sections of this notebook that
# don't use torchao directly -- so we upgrade it here, up front, before anything else
# has a chance to import peft against the stale version.
!pip install -q -U torchao
!pip install -q -U transformers peft accelerate
!pip install -q -U optimum-onnx onnxruntime onnx

print("Packages installed.")
print("If this is a fresh Colab runtime and any package above shows a 'RESTART RUNTIME'")
print("notice, do that now: Runtime -> Restart session, then Runtime -> Run all from the top.")

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,                      # rank -- see the lecture's worked example for what this controls
    lora_alpha=32,            # scaling factor applied to the LoRA update
    target_modules=["c_attn"],  # GPT-2's combined query/key/value attention projection
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

# The line above prints something like:
# trainable params: 147,456 || all params: 124,587,264 || trainable%: 0.1183
# -- compare this trainable percentage to the "100% trainable" that full fine-tuning would require.

### Preparing a small fine-tuning dataset

For this demo, we'll fine-tune GPT-2 to continue in a distinctive style — short, punchy motivational
one-liners. This is a small, illustrative dataset; a real project would use a larger one, but the
mechanics are identical regardless of dataset size.

In [ ]:
finetune_texts = [
    "Discipline beats motivation every single day.",
    "Small consistent steps build unstoppable momentum.",
    "The best time to start was yesterday. The next best time is now.",
    "Progress, not perfection, is the goal.",
    "Every expert was once a beginner who refused to quit.",
    "Focus on what you can control, and let go of the rest.",
    "Growth happens outside your comfort zone.",
    "A clear plan turns big goals into daily actions.",
    "Consistency compounds -- small efforts add up over time.",
    "Your future is built by what you do today, not what you intend to do tomorrow.",
] * 5   # repeat for a few more training examples given the short session time

print(f"Fine-tuning dataset: {len(finetune_texts)} short text examples")
print("Example:", finetune_texts[0])

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=48):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length", max_length=max_length,
            return_tensors="pt",
        )

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = item["input_ids"].clone()   # causal LM: predict the same sequence, shifted internally
        return item

finetune_dataset = TextDataset(finetune_texts, tokenizer)
finetune_loader = DataLoader(finetune_dataset, batch_size=4, shuffle=True)

print("Dataset ready:", len(finetune_dataset), "examples,", len(finetune_loader), "batches per epoch")

In [ ]:
# Fine-tune with LoRA -- notice this loop is structurally identical to Section 2's training loop.
lora_optimizer = torch.optim.AdamW(
    [p for p in lora_model.parameters() if p.requires_grad],  # only LoRA's trainable params
    lr=2e-4,
)

EPOCHS = 3
lora_model.train()

lora_losses = []
for epoch in range(EPOCHS):
    for batch in finetune_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = lora_model(**batch)     # forward pass + loss computed internally by the model
        loss = outputs.loss

        lora_optimizer.zero_grad()
        loss.backward()                    # backward pass
        lora_optimizer.step()              # optimizer step -- only updates LoRA's small matrices

        lora_losses.append(loss.item())

    print(f"Epoch {epoch+1}/{EPOCHS} | Last batch loss: {lora_losses[-1]:.4f}")

print("\nLoRA fine-tuning complete.")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(lora_losses)
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("GPT-2 + LoRA fine-tuning loss")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Compare generation before and after LoRA fine-tuning
lora_model.eval()
prompt = "The key to success is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output = lora_model.generate(
        **inputs, max_new_tokens=30, do_sample=True, temperature=0.8,
        pad_token_id=tokenizer.eos_token_id,
    )

print("GPT-2 + LoRA output (after fine-tuning on short motivational lines):")
print(tokenizer.decode(output[0], skip_special_tokens=True))
print("\n(With only a handful of training examples and 3 epochs, expect a stylistic shift")
print("toward shorter, punchier phrasing -- not a dramatic transformation.)")

### Save the LoRA adapter

Notice what actually gets saved: only the small LoRA matrices, not a full copy of GPT-2. This is the
direct, disk-visible proof of what "parameter-efficient" means in practice.

In [ ]:
lora_model.save_pretrained("gpt2-lora-adapter")

import os
adapter_size_mb = sum(
    os.path.getsize(os.path.join("gpt2-lora-adapter", f))
    for f in os.listdir("gpt2-lora-adapter")
) / (1024 * 1024)

print(f"LoRA adapter folder size: {adapter_size_mb:.2f} MB")
print("Compare this to GPT-2's full size (~500 MB in FP32) -- this is what 'save only the adapter' means.")

## 5. Quantization — Compress the Fine-Tuned Model

We'll merge the LoRA adapter back into the base model, then quantize it to INT8 and measure the real
difference in memory footprint.

> **A genuine gotcha worth knowing:** GPT-2 internally uses a layer type called `Conv1D` for its
> attention and feed-forward weights (a historical quirk from the original GPT-2 implementation),
> instead of the more common `nn.Linear`. Most quantization libraries — including the one we use below —
> specifically target `nn.Linear` layers and will silently skip `Conv1D` ones, giving you a "successful"
> quantization that changes nothing measurable. The fix is a short conversion step before quantizing,
> shown below — a small, real example of why checking *what actually changed*, not just whether a
> function ran without error, matters when optimizing a model.

In [ ]:
# Merge LoRA weights into the base model -- produces a single standalone model,
# no longer dependent on the separate adapter at inference time.
merged_model = lora_model.merge_and_unload()

# Move to CPU explicitly for this section: quantization and the CPU-timing comparisons
# below are specifically about CPU deployment, regardless of what device training used.
merged_model = merged_model.to("cpu")
merged_model.eval()

merged_model.save_pretrained("gpt2-finetuned-merged")
tokenizer.save_pretrained("gpt2-finetuned-merged")


def real_model_size_bytes(model):
    """Measure a model's true parameter storage size in bytes.

    Why this function exists: torchao's quantized tensor types (e.g. Int8Tensor) present
    an outer .dtype of float32 and .element_size() of 4 for backward-compatibility with
    code that doesn't know about them -- calling p.numel() * p.element_size() directly, or
    using transformers' built-in get_memory_footprint(), will report the ORIGINAL FP32 size
    even after quantization, because it's reading through that compatibility facade rather
    than the actual compressed storage underneath. This function looks at the real
    underlying storage tensors (qdata / scale / zero_point) when present.
    """
    total = 0
    for p in model.parameters():
        if hasattr(p, "qdata"):  # torchao quantized tensor subclass
            total += p.qdata.numel() * p.qdata.element_size()
            if hasattr(p, "scale") and p.scale is not None:
                total += p.scale.numel() * p.scale.element_size()
            if hasattr(p, "zero_point") and p.zero_point is not None:
                total += p.zero_point.numel() * p.zero_point.element_size()
        else:
            total += p.numel() * p.element_size()
    return total


fp32_footprint_mb = real_model_size_bytes(merged_model) / (1024 * 1024)
print(f"Merged fine-tuned model size (FP32, CPU): {fp32_footprint_mb:.2f} MB")

In [ ]:
# Step 1: Convert GPT-2's Conv1D weight layers to standard nn.Linear layers.
# This is purely a change of container -- the underlying weight values are preserved
# (Conv1D stores its weight transposed relative to nn.Linear, so we transpose it back).
import torch.nn as nn
import copy

def convert_conv1d_to_linear(module):
    """Recursively replace HuggingFace GPT-2's Conv1D layers with equivalent nn.Linear layers."""
    from transformers.pytorch_utils import Conv1D

    for name, child in module.named_children():
        if isinstance(child, Conv1D):
            # Conv1D.weight has shape (in_features, out_features) -- the transpose of nn.Linear's
            in_features, out_features = child.weight.shape
            linear = nn.Linear(in_features, out_features, bias=child.bias is not None)
            linear.weight.data = child.weight.data.T.contiguous()
            if child.bias is not None:
                linear.bias.data = child.bias.data
            setattr(module, name, linear)
        else:
            convert_conv1d_to_linear(child)
    return module

# merged_model is already on CPU (from the previous cell), so deepcopy + the newly-created
# nn.Linear replacements (which default to CPU) will all end up on the same device.
linear_model = convert_conv1d_to_linear(copy.deepcopy(merged_model))
linear_model.eval()

# Sanity check: converted model should produce (nearly) identical outputs to the original.
# Created on CPU to match both models above -- no .to(device) needed here.
test_inputs = tokenizer("Success comes from", return_tensors="pt")
with torch.no_grad():
    orig_logits = merged_model(**test_inputs).logits
    conv_logits = linear_model(**test_inputs).logits

max_diff = (orig_logits - conv_logits).abs().max().item()
print(f"Max output difference after Conv1D -> Linear conversion: {max_diff:.2e}")
print("(Should be extremely small -- this confirms the conversion preserved the model's behavior.)")

In [ ]:
# Step 2: Quantize the converted model to INT8 using torchao -- now it actually
# targets real nn.Linear layers, so the savings are measurable (with the right measurement --
# see the real_model_size_bytes() note in the previous cell).
from torchao.quantization import quantize_, Int8WeightOnlyConfig

quantized_model = copy.deepcopy(linear_model)
quantize_(quantized_model, Int8WeightOnlyConfig(version=2))

linear_fp32_footprint_mb = real_model_size_bytes(linear_model) / (1024 * 1024)
int8_footprint_mb = real_model_size_bytes(quantized_model) / (1024 * 1024)
reduction_pct = (1 - int8_footprint_mb / linear_fp32_footprint_mb) * 100

print(f"FP32 (Linear-converted) size:  {linear_fp32_footprint_mb:.2f} MB")
print(f"INT8 (quantized) size:         {int8_footprint_mb:.2f} MB")
print(f"Reduction: {reduction_pct:.1f}%")

if reduction_pct < 5:
    print("\n⚠️  Little or no measured reduction -- double check that Int8WeightOnlyConfig(version=2)")
    print("   actually applied (no errors above), and that real_model_size_bytes() from the previous")
    print("   cell is being used here rather than get_memory_footprint().")
else:
    print("\n✅ Quantization produced a real, measurable size reduction.")

# Confirm the quantized model still generates sensible text
with torch.no_grad():
    quant_out = quantized_model.generate(
        **test_inputs, max_new_tokens=20, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
print("\nQuantized model output:", tokenizer.decode(quant_out[0], skip_special_tokens=True))

In [ ]:
# Compare inference speed: FP32 (Linear-converted) vs. INT8 quantized, on CPU
import time

def time_generation(model, inputs, num_runs=5):
    model.eval()
    times = []
    for _ in range(num_runs):
        start = time.time()
        with torch.no_grad():
            _ = model.generate(
                **inputs, max_new_tokens=20, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        times.append(time.time() - start)
    return sum(times) / len(times)

fp32_time = time_generation(linear_model, test_inputs)
quant_time = time_generation(quantized_model, test_inputs)

print(f"FP32 (Linear) average generation time: {fp32_time:.3f} sec")
print(f"INT8 (quantized) average generation time: {quant_time:.3f} sec")
if quant_time < fp32_time:
    print(f"Speedup: {fp32_time/quant_time:.2f}x")
else:
    print("Note: on CPU, INT8 weight-only quantization primarily saves MEMORY, not always speed --")
    print("speed gains are typically clearer on hardware with native INT8 acceleration (many GPUs, some CPUs).")

### What just happened -- and why the two-step process mattered

- **The Conv1D → Linear conversion was necessary but, on its own, not sufficient.** GPT-2's original
  `Conv1D` layers are silently skipped by most `nn.Linear`-targeting quantization tools -- converting
  them first is a real prerequisite, verified above by checking that model outputs barely changed
  (a difference on the order of 1e-4, from floating-point rounding in the transpose, not a behavior
  change).
- **Whether the size reduction is visible depends on your runtime.** `torchao`'s INT8 tensor format is
  primarily built and optimized for GPU execution. On a CPU-only runtime, the measured memory footprint
  may not shrink even though quantization completed without error -- an honest example of a technique
  that's real and useful in its intended environment (GPU inference), but not guaranteed to show its
  effect everywhere you might run the same three lines of code.
- **Memory footprint, not disk file size, is what we measured**, using `get_memory_footprint()` rather
  than `save_pretrained` or `torch.save` -- serialized file size does not reliably reflect a quantized
  model's actual size for every tensor format, which is itself worth knowing before trusting a disk-size
  comparison as proof of compression.

In [ ]:
from optimum.onnxruntime import ORTModelForCausalLM

# export=True triggers on-the-fly conversion from the PyTorch checkpoint to ONNX format
onnx_model = ORTModelForCausalLM.from_pretrained(
    "gpt2-finetuned-merged", export=True,
)
onnx_model.save_pretrained("gpt2-onnx")

onnx_size_mb = sum(
    os.path.getsize(os.path.join("gpt2-onnx", f))
    for f in os.listdir("gpt2-onnx")
    if f.endswith(".onnx")
) / (1024 * 1024)

print(f"ONNX model size: {onnx_size_mb:.2f} MB")
print("(ONNX export alone doesn't compress the model the way quantization does --")
print(" its benefit is portability and runtime optimization, not smaller file size by itself.)")

In [ ]:
# Run inference with the ONNX model -- no PyTorch model object involved at generation time.
# Reuse the same test_inputs from the quantization section, so every version in this
# notebook (FP32, quantized, ONNX) is compared on the exact same input.
with torch.no_grad():
    onnx_output = onnx_model.generate(
        **test_inputs, max_new_tokens=20, do_sample=False,
    )

print("ONNX Runtime output:")
print(tokenizer.decode(onnx_output[0], skip_special_tokens=True))

In [ ]:
# Time ONNX Runtime inference the same way we timed FP32 and quantized PyTorch above
def time_onnx_generation(model, inputs, num_runs=5):
    times = []
    for _ in range(num_runs):
        start = time.time()
        _ = model.generate(**inputs, max_new_tokens=20, do_sample=False)
        times.append(time.time() - start)
    return sum(times) / len(times)

onnx_time = time_onnx_generation(onnx_model, test_inputs)

print(f"FP32 (Linear) average generation time:     {fp32_time:.3f} sec")
print(f"Quantized (INT8) average generation time:  {quant_time:.3f} sec")
print(f"ONNX Runtime average generation time:      {onnx_time:.3f} sec")

### Optional: quantize the ONNX model too

ONNX models can themselves be quantized — stacking ONNX Runtime's graph optimizations with reduced
precision, rather than choosing one or the other.

In [ ]:
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

quantizer = ORTQuantizer.from_pretrained(onnx_model)
qconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)

quantizer.quantize(save_dir="gpt2-onnx-quantized", quantization_config=qconfig)

onnx_quant_size_mb = sum(
    os.path.getsize(os.path.join("gpt2-onnx-quantized", f))
    for f in os.listdir("gpt2-onnx-quantized")
    if f.endswith(".onnx")
) / (1024 * 1024)

print(f"ONNX + quantized model size: {onnx_quant_size_mb:.2f} MB")
print(f"Size reduction vs. original ONNX: {(1 - onnx_quant_size_mb/onnx_size_mb) * 100:.1f}%")

## 7. Benchmark Comparison & Wrap-Up

Let's pull every measurement from this session into one comparison table — the same discipline the
lecture emphasized: always measure size, speed, *and* quality together, not resource savings in
isolation.

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Version": ["FP32 (PyTorch, Linear-converted)", "Quantized INT8 (PyTorch)", "ONNX Runtime", "ONNX + Quantized"],
    "Size / Footprint (MB)": [linear_fp32_footprint_mb, int8_footprint_mb, onnx_size_mb, onnx_quant_size_mb],
    "Avg. Generation Time (s)": [fp32_time, quant_time, onnx_time, None],
})
comparison["Size / Footprint (MB)"] = comparison["Size / Footprint (MB)"].round(1)
comparison["Avg. Generation Time (s)"] = comparison["Avg. Generation Time (s)"].round(3)

comparison

# Note: the FP32 and INT8 rows above are in-memory footprint (get_memory_footprint());
# the ONNX rows are on-disk file size (.onnx file) -- these are two different measurements
# of "how big is this model," not directly interchangeable, which is worth saying out loud
# if a sharp question comes from the room.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

versions = ["FP32\n(Linear)", "Quantized\n(INT8)", "ONNX\nRuntime"]
sizes = [linear_fp32_footprint_mb, int8_footprint_mb, onnx_size_mb]
times = [fp32_time, quant_time, onnx_time]

axes[0].bar(versions, sizes, color=["#2563eb", "#ea580c", "#16a34a"])
axes[0].set_title("Model Size / Footprint Comparison")
axes[0].set_ylabel("MB")

axes[1].bar(versions, times, color=["#2563eb", "#ea580c", "#16a34a"])
axes[1].set_title("Inference Time Comparison")
axes[1].set_ylabel("Seconds (avg. per generation)")

plt.tight_layout()
plt.show()

### What this session covered, end to end

1. **Training loop mechanics** — a full forward/loss/backward/optimizer-step loop on a tiny model,
   small enough to watch converge live
2. **PEFT / LoRA in practice** — freezing a 124M-parameter model and adapting it with a fraction of a
   percent of its parameters, with a real, measured trainable-parameter count
3. **Quantization** — compressing the fine-tuned model to INT8, with real measured size and speed
   numbers, not just claimed figures
4. **ONNX export** — decoupling the model from PyTorch for portable, runtime-optimized inference

### Extending this notebook on your own

- Try a larger `r` (LoRA rank) and observe the trainable-parameter-count and quality trade-off directly
- Try QLoRA — load the base model in 4-bit using `bitsandbytes`, then apply LoRA on top, for an even
  more memory-efficient fine-tuning setup
- Apply GPTQ or a static (calibration-based) quantization method, and compare accuracy retention
  against the dynamic quantization used here
- Deploy the ONNX model behind a lightweight API (e.g., FastAPI) and measure real request latency,
  not just generation time in-notebook

### Resources

- PEFT documentation: https://huggingface.co/docs/peft
- LoRA paper ("LoRA: Low-Rank Adaptation of Large Language Models"): https://arxiv.org/abs/2106.09685
- Optimum ONNX export guide: https://huggingface.co/docs/optimum-onnx
- ONNX Runtime: https://onnxruntime.ai

---

**Er. Abhishek Kumar Shukla**
Senior AI Developer, UniConverge Technologies Pvt. Ltd.
